# MRI Refacing Pipeline
This notebook implements a refacing pipeline for defaced MRI scans using atlas registration,
intensity normalisation, and Gaussian noise addition for realistic reconstruction.
* Atlas-based reconstruction: Uses the MNI ICBM152 T1 template atlas registered to the patient's space
* Intensity normalization: Matches atlas intensities to the patient scan for seamless blending
* Realistic noise addition: Adds Gaussian noise to simulate authentic MRI acquisition

In [10]:
# Import required libraries
import os
import nibabel as nib
import numpy as np
from scipy import ndimage
from time import time


## Noise Addition Function
Gaussian noise is added to simulate realistic MRI noise in the refaced region.

In [11]:
def add_gaussian_noise(image, sigma=15):
    """
    Add Gaussian noise (approximation for high SNR MRI)
    
    Parameters:
    -----------
    image : ndarray
        Input MRI image
    sigma : float
        Standard deviation of noise (higher = more noise)
        
    Returns:
    --------
    noisy_image : ndarray
        Image with added Gaussian noise
    """
    # Generate random Gaussian noise with mean=0 and std=sigma
    noise = np.random.normal(0, sigma, image.shape)
    
    # Add noise to image
    noisy_image = image + noise
    
    # Clip negative values (MRI intensities are non-negative)
    noisy_image = np.clip(noisy_image, 0, None)
    
    return noisy_image

## Step 1: Atlas Registration
Register the MNI atlas to the defaced patient scan using the transformation from skull-stripping.

In [12]:
# Define file paths
# Transformation file from skull-stripping registration step
transform_file = "./data/example_output_hdbet/output_skullstripped_registration/IXI002-Guys-0828-T1/hd_bet_dilated_IXI002-Guys-0828-T1.txt"

# MNI ICBM152 T1 atlas (moving image to be registered)
moving_image = "./data/icbm152_ext55_model_sym_2020_nifti/icbm152_ext55_model_sym_2020/mni_icbm152_t1_tal_nlin_sym_55_ext.nii"

# Defaced patient scan (reference space)
reference_image = "./data/example_output_hdbet/output_skullstripped_registration/IXI002-Guys-0828-T1/hd_bet_dilated_IXI002-Guys-0828-T1_masked.nii.gz"

# Output path for registered atlas
output_image = "atlas_registered_to_test.nii.gz"

# Paths to BRAINSTools executables
BRAINSresample_bin_path = "./BRAINSTools/BRAINSResample"

# Apply the transformation using BRAINSResample
# This warps the atlas into the patient's space using the inverse transform
os.system(f'"{BRAINSresample_bin_path}" ' +
          f'--inputVolume "{moving_image}" ' +
          f'--referenceVolume "{reference_image}" ' +
          f'--outputVolume "{output_image}" ' +
          f'--warpTransform "{transform_file}" ' +
          f'--inverseTransform ' +  # Use inverse transform to go from reference to moving
          f'--interpolationMode Linear')  # Linear interpolation for smooth result

0

## Step 2: Load Images and Masks
Load the reference (defaced) image, registered atlas, brain mask, and defacing mask.

In [13]:
# ============================================================================
# LOAD IMAGES
# ============================================================================
# Load the actual image data (not just masks)
reference_data = nib.load(reference_image).get_fdata()
output_data = nib.load(output_image).get_fdata()
print(f"Reference Image: {reference_data.shape}")
print(f"Output Image: {output_data.shape}")

# ============================================================================
# LOAD MASKS
# ============================================================================
# Brain mask: Defines the brain region that should NOT be modified
# (dilated version from skull-stripping to ensure full brain coverage)
brain_mask = nib.load("data/example_input_hdbet_processed/IXI002-Guys-0828-T1/hd_bet_dilated_IXI002-Guys-0828-T1.nii.gz").get_fdata()
brain_mask = np.where(brain_mask > 0, 1, 0)  # Binarize: 1 = brain, 0 = not brain
print(f"Brain mask: {brain_mask.shape}")

# Defacing mask: Defines the region that was removed by defacing
# We invert it because the original mask is 1=keep, 0=remove
defacing_mask = nib.load("data/example_output_hdbet/output_skullstripped_registration/IXI002-Guys-0828-T1/t1_mask_resampled.nii.gz").get_fdata().astype(int)
defacing_mask = (~defacing_mask.astype(bool)).astype(int)  # Invert: 1 = defaced region, 0 = kept region
print(f"Defacing mask: {defacing_mask.shape}")

print(f"Brain mask: {np.sum(brain_mask):,} voxels")
print(f"Defacing mask: {np.sum(defacing_mask):,} voxels")

# ============================================================================
# CHECK SHAPE COMPATIBILITY
# ============================================================================
# Ensure all arrays have the same shape (may need transposing)
if brain_mask.shape != defacing_mask.shape:
    if sorted(brain_mask.shape) == sorted(defacing_mask.shape):
        print("Shapes have same dimensions but different order - transposing defacing mask")
        # Transpose from (256, 256, 150) to (150, 256, 256)
        defacing_mask = np.transpose(defacing_mask, (2, 0, 1))
        print(f"Defacing mask shape after transpose: {defacing_mask.shape}")
    else:
        raise ValueError(f"Mask shapes don't match and can't be aligned: {brain_mask.shape} vs {defacing_mask.shape}")

# ============================================================================
# FIND OVERLAP REGION FOR INTENSITY NORMALIZATION
# ============================================================================
# Find intersection between brain mask and defacing mask
# This overlap region will be used for intensity calibration
intersection = brain_mask & defacing_mask
intersection_volume = np.sum(intersection)
print(f"Intersection: {intersection_volume:,} voxels")

print(f"\nShape verification:")
print(f"  Reference Image: {reference_data.shape}")
print(f"  Output Image: {output_data.shape}")
print(f"  Brain mask: {brain_mask.shape}")
print(f"  Defacing mask: {defacing_mask.shape}")

# Store ORIGINAL brain mask
brain_mask_original = brain_mask.copy()

# ============================================================================
# DILATE MASKS IF INTERSECTION IS TOO SMALL
# ============================================================================
if not (intersection_volume > 100 and np.sum(reference_data[intersection > 0]) != 0): 
    print(f"\nIntersection too small - dilating masks by ~5mm")
    
    # Get voxel spacing
    reference_nii = nib.load(reference_image)
    voxel_spacing = reference_nii.header.get_zooms()
    print(f"  Voxel spacing: {voxel_spacing} mm")
    
    # Calculate iterations
    min_spacing = min(voxel_spacing)
    iterations_5mm = int(np.ceil(5.0 / min_spacing))
    print(f"  Dilating by {iterations_5mm} iterations (~5mm)")
    
    # Dilate both masks
    brain_mask_dilated = ndimage.binary_dilation(brain_mask, iterations=iterations_5mm).astype(int)
    defacing_mask_dilated = ndimage.binary_dilation(defacing_mask, iterations=iterations_5mm).astype(int)
    
    # Update brain_mask to dilated version for intensity normalization
    brain_mask = brain_mask_dilated
    defacing_mask = defacing_mask_dilated
    
    # Recompute intersection with dilated masks
    intersection = brain_mask & defacing_mask
    intersection_volume = np.sum(intersection)
    print(f"  Intersection after dilation: {intersection_volume:,} voxels")

# Extract intensities (now using potentially dilated masks)
reference_intersection_intensities = reference_data[intersection > 0]
output_intersection_intensities = output_data[intersection > 0]


# Compute statistics for normalization
mean_reference_intersection = np.mean(reference_intersection_intensities)
std_reference_intersection = np.std(reference_intersection_intensities)
mean_output_intersection = np.mean(output_intersection_intensities)
std_output_intersection = np.std(output_intersection_intensities)

print(f"\nIntersection statistics:")
print(f"  Reference intensities in intersection: mean={mean_reference_intersection:.2f}, std={std_reference_intersection:.2f}")
print(f"  Output intensities in intersection: mean={mean_output_intersection:.2f}, std={std_output_intersection:.2f}")

Reference Image: (150, 256, 256)
Output Image: (150, 256, 256)
Brain mask: (150, 256, 256)
Defacing mask: (150, 256, 256)
Brain mask: 1,607,322 voxels
Defacing mask: 3,388,222 voxels
Intersection: 1,026 voxels

Shape verification:
  Reference Image: (150, 256, 256)
  Output Image: (150, 256, 256)
  Brain mask: (150, 256, 256)
  Defacing mask: (150, 256, 256)

Intersection too small - dilating masks by ~5mm
  Voxel spacing: (1.199997, 0.9375, 0.9375) mm
  Dilating by 6 iterations (~5mm)
  Intersection after dilation: 21,583 voxels

Intersection statistics:
  Reference intensities in intersection: mean=128.16, std=185.77
  Output intensities in intersection: mean=42.81, std=36.01


## Step 3: Intensity Normalization
Normalize the atlas intensities to match the patient scan using z-score normalization
based on statistics from the intersection region.

In [14]:
# ============================================================================
# INTENSITY NORMALIZATION (Z-SCORE TRANSFORMATION)
# ============================================================================
# Match mean and standard deviation of the atlas to the reference in the intersection region
# Formula: ((x - mean_src) / std_src) * std_tgt + mean_tgt
#
# This ensures:
# 1. The atlas has the same intensity range as the patient scan
# 2. Seamless blending at the boundary between original and refaced regions

normalized_output_data = ((output_data - mean_output_intersection) / std_output_intersection) * \
                         std_reference_intersection + mean_reference_intersection

print(f"\nIntensity Normalization:")
print(f"  Output (before): mean={mean_output_intersection:.2f}, std={std_output_intersection:.2f}")
print(f"  Reference (target): mean={mean_reference_intersection:.2f}, std={std_reference_intersection:.2f}")
print(f"  Output (after normalization): mean={np.mean(normalized_output_data[intersection > 0]):.2f}, "
      f"std={np.std(normalized_output_data[intersection > 0]):.2f}")
print(f"  Normalization successful - intensities match")


Intensity Normalization:
  Output (before): mean=42.81, std=36.01
  Reference (target): mean=128.16, std=185.77
  Output (after normalization): mean=128.16, std=185.77
  Normalization successful - intensities match


## Step 4: Merge and Add Noise
Fill the defaced region with the normalized atlas and add Gaussian noise for realism.

In [15]:
# ============================================================================
# MERGE REFERENCE AND NORMALIZED ATLAS
# ============================================================================
# Start with a copy of the reference (defaced) image
merged_data = reference_data.copy()

# Identify voxels to fill (where reference was defaced = 0)
mask_to_fill = (reference_data == 0)

print(f"\nMerging and Noise Addition:")
print(f"  Voxels to fill (defaced region): {np.sum(mask_to_fill):,}")

# ============================================================================
# ADD GAUSSIAN NOISE TO ATLAS
# ============================================================================
# Add noise to the normalized atlas to make it look more realistic
print(f"  Adding Gaussian noise to atlas before merging")
noisy_output = add_gaussian_noise(normalized_output_data, sigma=20)

# Fill the defaced region with the noisy, normalized atlas
merged_data[mask_to_fill] = noisy_output[mask_to_fill]

# ============================================================================
# SAVE THE REFACED IMAGE
# ============================================================================
# Save using the same affine transformation and header as the reference
merged_nii = nib.Nifti1Image(merged_data, affine=reference_nii.affine, header=reference_nii.header)
final_image = output_image.replace(".nii.gz", "_refaced.nii.gz")
nib.save(merged_nii, final_image)

print(f"  Merged image saved to {final_image}")


Merging and Noise Addition:
  Voxels to fill (defaced region): 5,042,716
  Adding Gaussian noise to atlas before merging
  Merged image saved to atlas_registered_to_test_refaced.nii.gz
